# Week 8 · Topic 2 — Reading Files from the Web



Not every dataset needs an API — sometimes it is a file sitting on the web,
waiting to be downloaded.



---
## 1. A "raw" GitHub URL serves the real file

`github.com/user/repo/blob/...` shows a formatted web page.
`raw.githubusercontent.com/user/repo/branch/...` gives you the actual file.

In [6]:
import requests

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
response = requests.get(url, timeout=10)
print(response.status_code)
print(response.text[:100])

200
sepal_length,sepal_width,petal_length,petal_width,species
5.1,3.5,1.4,0.2,setosa
4.9,3.0,1.4,0.2,set


### Your turn
Fetch that same URL and print how many characters `response.text` contains.

In [ ]:
# your code here


---
## 2. `requests` + `StringIO` — the reliable way into pandas

`StringIO` lets pandas treat downloaded text as if it were a local file. This
is the method to reach for by default.

In [4]:
import pandas as pd
from io import StringIO

df = pd.read_csv(StringIO(response.text))
print(df.head())
print(len(df), "rows")

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa
150 rows


### Your turn
Load the same URL into a DataFrame using `requests` + `StringIO`, and print its shape.

In [ ]:
# your code here



---
## 3. `pd.read_csv(url)` directly — shorter, not always reliable

pandas can fetch a URL itself. It often works — but it uses your system's own
certificate setup, not `requests`' bundled one, and can fail with an SSL error
on some machines even when `requests.get()` on the same URL is fine. Section 2's
method always works instead.

In [ ]:
# df = pd.read_csv(url)
# SSL: CERTIFICATE_VERIFY_FAILED  <- can happen here even though
#                                    requests.get(url) worked above

### Your turn
Try `pd.read_csv(url)` directly on your own machine. Does it work, or do you see the SSL error?

In [ ]:
# your code here

---
## 4. Kaggle, and your own hosted data

Kaggle needs an account and an API token (`pip install kaggle`, then
`kaggle datasets download -d owner/name`) — not a single `requests.get()`. But
once it is a `.csv` on disk, everything from here works the same. A CSV you
commit to your **own** GitHub repo works exactly like Topic 2's example — same
mechanism, nothing special about who owns it.

Step 1: Get your token from Kaggle
- Go to [kaggle.com](https://www.kaggle.com/) and log in (create a free account if you don’t have one).
- Click your profile picture (top right) → Settings.
- Scroll to the API section and click Create New Token.
- A file called kaggle.json downloads to your Downloads folder.- 

Step 2: Move the token into a folder called .kaggle

##### On a Mac: open the Terminal app and paste these two lines, pressing Enter after each:

```
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
```


##### On Windows: open Command Prompt (press the Windows key, type cmd, press Enter) and paste these two lines, pressing Enter after each:

```
mkdir %USERPROFILE%\.kaggle
move %USERPROFILE%\Downloads\kaggle.json %USERPROFILE%\.kaggle\

```



Step 3: Install the library

In Terminal:

```
pip install "kagglehub[pandas-datasets]"
```

In [11]:
import kagglehub
import pandas as pd
import os

# Step 1: Download the dataset (uses your token automatically)
path = kagglehub.dataset_download("uciml/iris")
print("Path to dataset files:", path)

# Step 2: See which files were downloaded
print(os.listdir(path))

# Step 3: It's now a CSV on disk, so load it normally
df = pd.read_csv(os.path.join(path, "Iris.csv"))
print(df.head())

100%|██████████| 3.60k/3.60k [00:00<00:00, 997kB/s]

Extracting files...
Path to dataset files: /Users/Anna29/.cache/kagglehub/datasets/uciml/iris/versions/2
['database.sqlite', 'Iris.csv']
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa


### Your turn
Check the dataset you loaded in section 2 against the checklist: 50+ rows, 4+ columns, one genuinely messy text column.

In [ ]:
# your code here

---
## 5. Nested JSON: `pd.json_normalize()`

Many APIs return records that contain other records. A plain `pd.DataFrame(records)` would leave a whole dictionary inside one
cell. `pd.json_normalize()` flattens the nesting into separate columns, named with a dot: `properties.mag` means "the `mag` value
inside `properties`".

Your project's `make_table` function in `fetch_data.py` uses this for every JSON dataset.

In [ ]:
import pandas as pd

records = [
    {"id": "a1", "properties": {"mag": 4.5, "place": "Chile", "time": 1700000000000}},
    {"id": "a2", "properties": {"mag": None, "place": "Japan", "time": 1700003600000}},
    {"id": "a3", "properties": {"mag": 5.1, "place": "Peru", "time": 1700007200000}},
]

df = pd.json_normalize(records)
print(df.columns.tolist())
df

Dots in column names cause trouble later (SQL treats a dot as a separator). The simplest fix is to ask `json_normalize` for an underscore
instead, with `sep="_"`:

In [ ]:
df = pd.json_normalize(records, sep="_")
print(df.columns.tolist())          # properties_mag, properties_place ...

### Your turn
Add a fourth record whose `properties` has an extra key, `"depth": 10`, and run `json_normalize()` again. What happens to the other rows?

In [ ]:
# your code here

---
## 5b. A column that holds a list: `.apply()`

Sometimes one cell holds a **list**. The earthquake API stores the position as `[longitude, latitude, depth]`. `.apply()` runs a small function on every
value of a column, and `lambda` is a one-line function with no name:

```python
df["depth_km"] = df["coordinates"].apply(lambda coords: coords[2])
```

Read it as: *for each `coords` in the column, give back item number 2.*

In [ ]:
quakes = pd.DataFrame({"place": ["Chile", "Japan"], "coordinates": [[-70.1, -33.4, 12.0], [139.7, 35.7, 30.5]]})
quakes["depth_km"] = quakes["coordinates"].apply(lambda coords: coords[2])
quakes

### Your turn
Add a `longitude` column the same way (it is item number 0).

In [ ]:
# your code here

---
## 6. Tidying a table: choose and rename columns

Fetched data usually has more columns than you need, with awkward names. Three tools do most of the work:

| Task | Code |
|---|---|
| rename some columns | `df.rename(columns={"old": "new"})` |
| keep only some columns, in an order you choose | `df[["place", "mag", "time"]]` |
| find the columns of one kind | `df.select_dtypes("number")`, and `df.select_dtypes(exclude="number")` for everything else |

`select_dtypes` is how the lab checks that your data has at least one number column and one text column.

In [ ]:
df = df.rename(columns={"properties_mag": "mag", "properties_place": "place", "properties_time": "time"})
df = df[["place", "mag", "time"]]

print("number columns:", df.select_dtypes("number").columns.tolist())
print("text columns:  ", df.select_dtypes("object").columns.tolist())
df

### Your turn
Rename `place` to `location`, then print the text columns again.

In [ ]:
# your code here

---
## 7. Tidying a table: missing values, dates and ranges

| Task | Code |
|---|---|
| count missing values in each column | `df.isna().sum()` |
| drop rows where a column is missing | `df.dropna(subset=["mag"])` |
| turn milliseconds since 1970 into dates | `pd.to_datetime(df["time"], unit="ms")` |
| write a date as text in a format you choose | `df["time"].dt.strftime("%Y-%m-%d %H:%M:%S")` |
| keep rows whose value lies inside a range | `df[df["mag"].between(4, 5)]` |

`dropna(subset=...)` only looks at the columns you name. Plain `dropna()` removes any row with a gap anywhere, which can delete
most of a wide table. Dates are saved as text (`strftime`) because a database stores dates as text.

In [ ]:
print(df.isna().sum())

df = df.dropna(subset=["mag"])
df["time"] = pd.to_datetime(df["time"], unit="ms").dt.strftime("%Y-%m-%d %H:%M:%S")
print(df)

print(df[df["mag"].between(4, 5)])

### Your turn
Use `between()` to keep only the rows with a magnitude from 5 to 6. How many are left?

In [ ]:
# your code here

---
## 8. Where files live: `pathlib`

A file location written as a string (`"data/raw.csv"`) depends on the folder the program was started from. A `Path` built from the
location of the code itself does not. This is how your project's `fetch_data.py` finds its `DATA` folder, wherever you run it from.

| Code | Meaning |
|---|---|
| `Path(__file__).parent` | the folder this `.py` file is in (in a script) |
| `folder / "raw.csv"` | join a folder and a file name, correctly on every operating system |
| `path.exists()` | does it exist? |
| `path.parent.mkdir(parents=True, exist_ok=True)` | create the folder if it is missing |

In [ ]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())          # in a script you would use Path(__file__).parent
target = folder / "data" / "tidy.csv"

print(target.exists())
target.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(target, index=False)
print(target.exists(), "|", target.name, "|", target.parent.name)

### Your turn
Print `target.suffix`, then use `pd.read_csv(target)` to read the file back.

In [ ]:
# your code here

`examples/03_nested_json_and_tidying.py` puts sections 5 to 8 together in one script. Your lab's `tidy()` method is this same sequence applied to your own data.

---
## Topic 2 Challenge

Download a real CSV of your choice (or reuse `iris.csv`) using `requests` +
`StringIO`, and print its column names and row count.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | downloading a Kaggle dataset for analysis |
| **Cyber Security** | downloading a public breach or phishing-domain list |
| **IT** | downloading a public device/config inventory file |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| using the `blob` (web page) URL instead of `raw` | HTML, not data | swap `blob` for `raw`, use `raw.githubusercontent.com` |
| `pd.read_csv(url)` failing with an SSL error | `SSL: CERTIFICATE_VERIFY_FAILED` | use `requests` + `StringIO` instead |
| forgetting `StringIO` and passing `response.text` straight to `read_csv` | `TypeError` | wrap it: `StringIO(response.text)` |
| assuming Kaggle is one line like a raw URL | stuck without an account/token | Kaggle needs `pip install kaggle` and a token first |
| `dropna()` with no `subset` | most rows disappear from a wide table | name the columns that matter: `subset=[...]` |
| a column name with a dot | `KeyError` when you write `df["properties.mag"]` wrongly, or SQL errors later | replace dots with `_` straight after `json_normalize` |

## Go deeper

`examples/01_downloading_a_csv.py` — raw URLs, requests + StringIO, the SSL gotcha
`examples/02_kaggle_and_your_own_data.py` — Kaggle's flow, your own hosted data, the checklist
Docs: [requests documentation](https://requests.readthedocs.io/en/latest/)

---
## Recap

| You can now | Using |
|---|---|
| get the real file, not a web page | a `raw.githubusercontent.com` URL |
| load it into pandas reliably | `requests` + `StringIO` |
| know the shortcut's risk | `pd.read_csv(url)` can fail with an SSL error |
| check a dataset is worth using | the checklist: rows, columns, a messy column |
| flatten nested JSON | `pd.json_normalize(records, sep="_")` |
| run a function on a column | `df["x"].apply(lambda v: ...)` |
| rename and choose columns | `df.rename(columns={...})`, `df[[...]]` |
| find number or text columns | `df.select_dtypes("number")` |
| handle gaps, dates and ranges | `dropna(subset=...)`, `to_datetime`, `strftime`, `between` |
| build file locations | `Path(__file__).parent / "data"` |

**Next:** Topic 3 is an optional extra (finding a pattern in text that has no separator to split on). Your lab comes next.